In [64]:
dir_path = "/home/chanhui-lee/text-mol/MolCA/data/multi_task_0927/raw"
test_files = [
    "smol-forward_synthesis_subtask-0_test.pth",
    "smol-molecule_generation_subtask-0_test.pth",
]

# load the testsets
import torch
import os
import numpy as np
import pandas as pd
import random
import selfies as sf

def load_testset(dir_path, test_files):

    testset = []
    for f in test_files:
        list_of_instances = torch.load(os.path.join(dir_path, f))
        list_of_dicts = []
        for instance in list_of_instances:
            dict_instance = {
                "graph": instance[0],
                "label_selfies": instance[1],
                "input_mol_selfies": instance[2],
                "task_subtask_pair": instance[3],
                "instruction": instance[4],
                "conditions": None,
            }
            label_selfies = instance[1].replace("<SELFIES>", "").replace("</SELFIES>", "")
            label_smiles = sf.decoder(label_selfies)
            input_mol_selfies = instance[2].replace("<SELFIES>", "").replace("</SELFIES>", "")
            if "None" in input_mol_selfies:
                input_mol_smiles = input_mol_selfies
            else:
                input_mol_smiles = sf.decoder(input_mol_selfies)

            dict_instance.update({
                "label_selfies": label_selfies,
                "label_smiles": label_smiles,
                "input_mol_selfies": input_mol_selfies,
                "input_mol_smiles": input_mol_smiles,
            })
            list_of_dicts.append(dict_instance)
        random.shuffle(list_of_dicts)
        testset.append(list_of_dicts[:1000])

    return testset

test_paths = [os.path.join(dir_path, f) for f in test_files]
testset = load_testset(dir_path, test_files)
len(testset[0]), len(testset[1])

(1000, 1000)

In [65]:
testset[0][0], testset[1][0]

({'graph': Data(x=[33, 9], edge_index=[2, 70], edge_attr=[70, 3]),
  'label_selfies': '[N][#C][C][=C][C][=C][C][N][Branch2][Ring1][C][C][=Branch1][C][=O][N][C][C][C][C][C][=C][C][=C][C][=C][Ring1][=Branch1][C][C][Ring2][Ring1][C][=C][Ring2][Ring1][=Branch1]',
  'input_mol_selfies': '[C][=C][C][=C][C][N][C][C][Ring1][Branch1][=C][Ring1][=Branch2].[C][O][C][=Branch1][C][=O][C][=C][C][=C][Branch1][Ring2][N][=C][=O][C][=C][Ring1][=Branch2].[N][#C][C][=C][C][=C][C][N][C][C][Ring1][Branch1][=C][Ring1][=Branch2]',
  'task_subtask_pair': 'smol-forward_synthesis/smol-forward_synthesis',
  'instruction': '<INPUT> Given the above reactants and reagents, what could be a probable product of their reaction?',
  'conditions': None,
  'label_smiles': 'N#CC1=CC=C2CN(C(=O)NCCCCC3=CC=CC=C3)CC2=C1',
  'input_mol_smiles': 'C1=CC=C2CNCC2=C1.COC(=O)C3=CC=C(N=C=O)C=C3.N#CC4=CC=C5CNCC5=C4'},
 {'graph': Data(x=[2, 9], edge_index=[2, 2], edge_attr=[2, 3]),
  'label_selfies': '[C][C][Branch1][C][C][=C][C][C][/C][

In [66]:
# save the test_data_1k
test_files_1k = [
    "smol-forward_synthesis_subtask-0_test_1k.pth",
    "smol-molecule_generation_subtask-0_test_1k.pth",
]
os.makedirs(os.path.join(dir_path, "molgen_poc"), exist_ok=True)

for f in test_files_1k:
    for i, test in enumerate(testset):
        torch.save(test, os.path.join(dir_path, "molgen_poc", f))